# Descarga de datos: imágenes Sentinel-1 originales de lluvias en ciudades

Descarga **solo las imágenes de radar tal como las entrega el satélite** (Sentinel-1 GRD, nivel 1), tomadas el día de una
lluvia o en los días siguientes, mientras el agua puede seguir empozada. No usa datasets preprocesados: la calibración y el
resto los hace el notebook 01.

* **Fuente:** Microsoft Planetary Computer (colección `sentinel-1-grd`, sin cuenta).
* **Formato:** la estructura original `.SAFE` que lee SNAP: `measurement/*.tiff` (VV y VH), `annotation/**/*.xml`
  (anotación, calibración y ruido) y `manifest.safe`.
* **Destino:** `Data/crudos/<ciudad>/<escena>.SAFE`.
* **Días secos de referencia (sección 1.1):** 15 escenas de Lima sin lluvia (10 de 2026 y 5 de 2016-2017) en
  `Data/referencia_seca/lima/`, solo para comparar (notebook 01, secciones 6.2 a 6.4); no se usan para entrenar.
* **NISAR (sección 5):** imágenes de radar de 5 m en banda L (RSLC, nivel 1) de Lima Sur, del archivo de la NASA.

## 1. Lluvias y escenas

Cada fila es una lluvia confirmada en la prensa y la pasada del satélite de ese día o de los días siguientes, cuando el
agua todavía puede estar empozada (05:38-05:48 o 18:33 hora de Lima).

| Descartado | Motivo |
|---|---|
| Kuwait (nov 2018) | las pasadas son *antes* de las lluvias fuertes |
| Doha (oct 2018), Yeda (nov 2022) | la primera pasada es 5-7 días después; el agua ya se había ido |
| Lima: 21 feb 2023, 2 feb 2026 | solo hubo lluvia en la sierra o un aviso, sin reporte de lluvia en la ciudad |
| Lima: 12 feb 2025 | la siguiente pasada fue más de un día después |

In [ ]:
import re, datetime as dt, xml.etree.ElementTree as ET
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np, pandas as pd, requests, rasterio, matplotlib.pyplot as plt

cajas = {"lima": (-77.15, -12.30, -76.85, -11.85), "dubai": (55.15, 25.05, 55.50, 25.38),
         "karachi": (66.95, 24.82, 67.20, 25.05), "el_cairo": (31.15, 29.95, 31.45, 30.15),
         "mascate": (58.30, 23.55, 58.60, 23.63)}
huso = {"lima": -5, "dubai": 4, "karachi": 5, "el_cairo": 2, "mascate": 4}

eventos = pd.DataFrame([
    ("lima", "Niño costero: huaicos y desbordes en Lima Este (15-17 mar 2017)", "S1B_IW_GRDH_1SDV_20170317T103823_20170317T103848_004747_0084AE"),
    ("lima", "Lluvia de verano del 22 ene 2018 (Panamericana)", "S1B_IW_GRDH_1SDV_20180123T103835_20180123T103900_009297_010AD9"),
    ("lima", "Lluvia de trasvase del 18 mar 2019 (Senamhi)", "S1B_IW_GRDH_1SDV_20190319T103841_20190319T103906_015422_01CE20"),
    ("lima", "Ciclón Yaku: lluvias en Lima Sur y Este (7-16 mar 2023)", "S1A_IW_GRDH_1SDV_20230316T103938_20230316T104003_047668_05B9BA"),
    ("lima", "Lluvias de horas del 2-3 ene 2025 (La República)", "S1A_IW_GRDH_1SDV_20250104T103939_20250104T104004_057293_070C8F"),
    ("lima", "Aguacero de la noche del 9 ene 2026 (Trome)", "S1C_IW_GRDH_1SDV_20260110T104626_20260110T104703_005840_00BB1D"),
    ("lima", "Lluvias del 9-11 ene 2026 (emergencia DS 003-2026-PCM)", "S1A_IW_GRDH_1SDV_20260111T103929_20260111T103954_062718_07DD42"),
    ("lima", "Lluvia del 15 ene 2026 en Cercado, Rímac, Lince, Miraflores (Expreso)", "S1A_IW_GRDH_1SDV_20260116T104748_20260116T104808_062791_07E029"),
    ("lima", "DANA Aníbal: lluvia del 16 ago 2026 (VMT, VES)", "S1C_IW_GRDH_1SDV_20260817T233316_20260817T233341_009041_011F2C"),
    ("lima", "Lloviznas del 1-2 oct 2026 (baipás Nueva Esperanza inundado)", "S1C_IW_GRDH_1SDV_20261003T103835_20261003T103900_009719_0135C0"),
    ("lima", "Llovizna intensa del 22 oct 2026 (San Borja, SJL, Puente Piedra)", "S1D_IW_GRDH_1SDV_20261022T233336_20261022T233401_005134_009BB8"),
    ("lima", "Lluvias del 3-5 nov 2026 (65 puntos críticos)", "S1D_IW_GRDH_1SDV_20261103T233335_20261103T233400_005309_00A1B6"),
    ("lima", "Lluvias del 3-9 nov 2026", "S1C_IW_GRDH_1SDV_20261108T103834_20261108T103859_010244_014762"),
    ("lima", "Lluvia del 9 nov 2026 (baipás Nueva Esperanza inundado otra vez)", "S1C_IW_GRDH_1SDV_20261109T233316_20261109T233341_010266_01480F"),
    ("dubai", "Tormenta récord del 16 abr 2024; agua estancada en Sharjah", "S1A_IW_GRDH_1SDV_20240424T142514_20240424T142539_053577_068115"),
    ("karachi", "Lluvia récord del 27 ago 2020", "S1A_IW_GRDH_1SDV_20200828T012629_20200828T012657_034100_03F5A0"),
    ("karachi", "Lluvias monzónicas del 24-25 jul 2022", "S1A_IW_GRDH_1SDV_20220725T012639_20220725T012707_044250_05480F"),
    ("el_cairo", "Tormenta del 12 mar 2020 (calles inundadas)", "S1B_IW_GRDH_1SDV_20200313T035201_20200313T035225_020668_0272DA"),
    ("mascate", "Lluvias del 14 abr 2024 en Omán", "S1A_IW_GRDH_1SDV_20240414T140829_20240414T140854_053431_067B51"),
], columns=["ciudad", "lluvia", "escena"])
eventos["hora_local"] = [dt.datetime.strptime(s[17:32], "%Y%m%dT%H%M%S") + dt.timedelta(hours=huso[c])
                         for s, c in zip(eventos.escena, eventos.ciudad)]
eventos[["ciudad", "hora_local", "lluvia"]]

### 1.1 Días secos de referencia (Lima)

Para saber si algo es agua de la lluvia hay que ver cómo se ve ese lugar un día sin lluvia. Se eligieron 5 pasadas de la
mañana (05:38) y 5 de la tarde (18:33) de 2026, los mismos pases que las lluvias, con a lo más 0.5 mm en cuatro días
(precipitación ERA5 en cuatro puntos de Lima; 2026 tuvo garúa casi todo el año). Se descartaron las de Sentinel-1D del 30 y
31 de mayo: toda la ciudad sale 1.2 dB más oscura que en las demás fechas (calibración del satélite nuevo).

Para las lluvias de 2017-2019 se agregan 5 pasadas de la mañana de la misma órbita: septiembre de 2016 (la única anterior al
Niño costero con VV y VH y sin lluvia) y mayo a agosto de 2017. Entre diciembre de 2016 y abril de 2017 llovió casi siempre.

In [ ]:
secos = pd.DataFrame({"escena": [
    "S1C_IW_GRDH_1SDV_20260318T103840_20260318T103905_006817_00DC86",
    "S1A_IW_GRDH_1SDV_20260417T103926_20260417T103951_064118_0811AE",
    "S1A_IW_GRDH_1SDV_20260429T103926_20260429T103951_064293_081834",
    "S1A_IW_GRDH_1SDV_20260523T103925_20260523T103950_064643_0824A7",
    "S1D_IW_GRDH_1SDV_20260717T103845_20260717T103910_003712_006A56",
    "S1A_IW_GRDH_1SDV_20260418T233415_20260418T233440_064140_081287",
    "S1A_IW_GRDH_1SDV_20260430T233415_20260430T233440_064315_081909",
    "S1C_IW_GRDH_1SDV_20260506T233313_20260506T233338_007539_00F4FA",
    "S1C_IW_GRDH_1SDV_20260712T233314_20260712T233339_008516_010DC8",
    "S1D_IW_GRDH_1SDV_20260928T233335_20260928T233400_004784_008F8F",
    "S1A_IW_GRDH_1SDV_20160912T103911_20160912T103936_013018_0149E6",
    "S1B_IW_GRDH_1SDV_20170516T103826_20170516T103851_005622_009D8D",
    "S1B_IW_GRDH_1SDV_20170609T103832_20170609T103857_005972_00A7B2",
    "S1B_IW_GRDH_1SDV_20170703T103833_20170703T103858_006322_00B1DF",
    "S1B_IW_GRDH_1SDV_20170820T103836_20170820T103901_007022_00C5F6"]})
secos["hora_local"] = [dt.datetime.strptime(s[17:32], "%Y%m%dT%H%M%S") - dt.timedelta(hours=5) for s in secos.escena]
puntos = {"centro": (-12.06, -77.04), "sur": (-12.18, -76.94), "este": (-12.00, -76.99), "norte": (-11.90, -77.06)}
lluvia_dia = pd.concat([pd.Series(r["daily"]["precipitation_sum"], index=pd.to_datetime(r["daily"]["time"]), name=n)
                        for n, r in ((n, requests.get("https://archive-api.open-meteo.com/v1/archive",
                                                      params={"latitude": la, "longitude": lo, "start_date": "2016-09-01",
                                                              "end_date": "2026-10-01", "daily": "precipitation_sum"}).json())
                                     for n, (la, lo) in puntos.items())], axis=1).max(axis=1)
secos["lluvia_4_dias_mm"] = [lluvia_dia[h.normalize() - pd.Timedelta(days=3):h.normalize()].max() for h in secos.hora_local]
secos

## 2. Descarga

Para cada escena se baja su `manifest.safe` y, con los nombres que lista, las imágenes VV y VH y sus XML. Lo que ya existe no
se vuelve a bajar; lo que se está bajando queda como `.parcial`. Son unos 23 GB de lluvias y 15 GB de días secos.

> En los productos procesados desde 2022 el asset `schema-product` de Planetary Computer apunta al XML de interferencias (RFI)
> y no a la anotación; con ese archivo SNAP no abre la escena. Por eso los XML se toman de la carpeta del producto.

In [ ]:
stac = "https://planetarycomputer.microsoft.com/api/stac/v1"
token = requests.get("https://planetarycomputer.microsoft.com/api/sas/v1/token/sentinel-1-grd").json()["token"]


def url_de(item, nombre):
    base = item["assets"]["schema-calibration-vv"]["href"].rsplit("/annotation/", 1)[0]
    pol = re.search(r"-(vv|vh)-", nombre).group(1)
    if nombre.startswith("measurement/"):
        return item["assets"][pol]["href"]
    if nombre.startswith("annotation/calibration/calibration-"):
        return f"{base}/annotation/calibration/calibration-iw-{pol}.xml"
    if nombre.startswith("annotation/calibration/noise-"):
        return f"{base}/annotation/calibration/noise-iw-{pol}.xml"
    if nombre.startswith("annotation/rfi/"):
        return f"{base}/annotation/rfi/rfi-iw-{pol}.xml"
    return f"{base}/annotation/iw-{pol}.xml"


def bajar(url, ruta):
    if ruta.exists():
        return
    ruta.parent.mkdir(parents=True, exist_ok=True)
    parcial = ruta.with_suffix(ruta.suffix + ".parcial")
    with requests.get(url + "?" + token, stream=True) as r, open(parcial, "wb") as f:
        r.raise_for_status()
        for bloque in r.iter_content(8 << 20):
            f.write(bloque)
    parcial.rename(ruta)


archivos = []
destinos = [(e.escena, f"Data/crudos/{e.ciudad}") for e in eventos.itertuples()]
destinos += [(s, "Data/referencia_seca/lima") for s in secos.escena]
for escena, carpeta in destinos:
    item = requests.get(f"{stac}/collections/sentinel-1-grd/items/{escena}").json()
    safe = Path(f"{carpeta}/{escena}.SAFE")
    safe.mkdir(parents=True, exist_ok=True)
    manifest = requests.get(item["assets"]["safe-manifest"]["href"] + "?" + token).content
    (safe / "manifest.safe").write_bytes(manifest)
    for o in ET.fromstring(manifest).iter("fileLocation"):
        nombre = o.get("href").lstrip("./")
        if nombre.startswith(("measurement/", "annotation/")) and re.search(r"-(vv|vh)-", nombre):
            archivos.append((url_de(item, nombre), safe / nombre))
with ThreadPoolExecutor(6) as ex:
    list(ex.map(lambda a: bajar(*a), archivos))
print(len(archivos), "archivos")

### Compresión que SNAP pueda leer

Planetary Computer guarda las imágenes de 2024 en adelante comprimidas con ZSTD, que SNAP no lee. Se recomprimen **sin
pérdida** a DEFLATE y se comprueba fila por fila que los valores no cambian.

In [ ]:
import rasterio.shutil
from rasterio.windows import Window


def a_deflate(tiff):
    with rasterio.open(tiff) as f:
        if f.profile.get("compress", "").lower() != "zstd":
            return False
    copia = tiff.with_suffix(".deflate.tiff")
    rasterio.shutil.copy(tiff, copia, driver="GTiff", COMPRESS="DEFLATE", TILED="YES", BIGTIFF="IF_SAFER", NUM_THREADS="ALL_CPUS")
    with rasterio.open(tiff) as a, rasterio.open(copia) as b:
        for i in range(0, a.height, 2048):
            w = Window(0, i, a.width, min(2048, a.height - i))
            assert np.array_equal(a.read(1, window=w), b.read(1, window=w))
    copia.replace(tiff)
    return True


with ThreadPoolExecutor(4) as ex:
    print(sum(ex.map(a_deflate, sorted(Path("Data").glob("*/*/*.SAFE/measurement/*.tiff")))), "imágenes recomprimidas")

In [ ]:
filas = []
for safe in sorted(Path("Data").glob("*/*/*.SAFE")):
    necesarios = [o.get("href").lstrip("./") for o in ET.parse(safe / "manifest.safe").iter("fileLocation")]
    necesarios = [n for n in necesarios if n.startswith(("measurement/", "annotation/"))]
    tiffs = list(safe.glob("measurement/*.tiff"))
    filas.append({"tipo": "seco" if safe.parent.parent.name == "referencia_seca" else "lluvia",
                  "ciudad": safe.parent.name, "escena": safe.stem[17:32],
                  "archivos": f"{sum((safe / n).exists() for n in necesarios)}/{len(necesarios)}",
                  "GB": sum(t.stat().st_size for t in tiffs) / 1e9})
inventario = pd.DataFrame(filas)
print(f"{len(inventario)} escenas · {inventario.GB.sum():.1f} GB")
inventario.round(2)

## 3. Cómo se ven las imágenes originales

Un `.tiff` original guarda números digitales en **geometría de radar** (filas = tiempo de vuelo, columnas = distancia al
satélite), por eso aparece girado o en espejo respecto del mapa. Se recorta la ciudad con los puntos de control del archivo.

In [ ]:
from rasterio.transform import from_gcps
from rasterio.windows import from_bounds

safes = sorted(Path("Data/crudos").glob("*/*.SAFE"), key=lambda s: s.stem[17:32])
fig, ejes = plt.subplots(int(np.ceil(len(safes) / 5)), 5, figsize=(16, 3.4 * np.ceil(len(safes) / 5)))
for ax, safe in zip(ejes.flat, safes):
    with rasterio.open(next(safe.glob("measurement/*-vv-*.tiff"))) as f:
        v = from_bounds(*cajas[safe.parent.name], transform=from_gcps(f.gcps[0])).round_offsets().round_lengths()
        v = v.intersection(Window(0, 0, f.width, f.height))
        dn = f.read(1, window=v, out_shape=(int(v.height // 4), int(v.width // 4))).astype("float32")
    db = 20 * np.log10(np.maximum(dn, 1))
    ax.imshow(db, cmap="gray", vmin=np.percentile(db, 2), vmax=np.percentile(db, 98))
    ax.set_title(f"{safe.parent.name} {safe.stem[17:25]}", fontsize=8)
for ax in ejes.flat:
    ax.axis("off")
plt.tight_layout()

## 4. ¿Hay imagen óptica (Sentinel-2) del mismo día?

Para marcar el agua se buscó Sentinel-2 del mismo día local y se midió la nubosidad sobre la ciudad con su capa de
clasificación (SCL). Cuando llueve la ciudad está nublada (por eso se usa radar), así que la etiqueta se saca del propio radar
de ese día (notebook 01, sección 2.4).

In [ ]:
from shapely.geometry import box, shape
from rasterio.warp import transform_bounds

token_s2 = requests.get("https://planetarycomputer.microsoft.com/api/sas/v1/token/sentinel-2-l2a").json()["token"]
filas = []
for e in eventos.itertuples():
    inicio = dt.datetime.combine(e.hora_local.date(), dt.time()) - dt.timedelta(hours=huso[e.ciudad])
    items = requests.post(f"{stac}/search", json={"collections": ["sentinel-2-l2a"], "bbox": cajas[e.ciudad],
                          "datetime": f"{inicio:%Y-%m-%dT%H:%M:%SZ}/{inicio + dt.timedelta(days=1):%Y-%m-%dT%H:%M:%SZ}"}).json()["features"]
    nubes = []
    for item in items:
        with rasterio.open(item["assets"]["SCL"]["href"] + "?" + token_s2) as f:
            scl = f.read(1, window=from_bounds(*transform_bounds("EPSG:4326", f.crs, *cajas[e.ciudad]), f.transform),
                         boundless=True, fill_value=0)
        if (scl > 0).any():
            nubes.append(np.isin(scl[scl > 0], [3, 8, 9, 10]).mean())
    filas.append({"ciudad": e.ciudad, "dia_local": e.hora_local.date(),
                  "sentinel2_mismo_dia": f"nubes {min(nubes):.0%}" if nubes else "no hubo pasada"})
pd.DataFrame(filas)

## 5. NISAR: radar de 5 m en banda L

NISAR (NASA-ISRO) publica sus imágenes en el archivo de la NASA (ASF). Se bajan las **RSLC** (nivel 1: números complejos
en geometría de radar, sin geocodificar) de dos pistas que cubren Lima Sur, las mismas fechas que usa el notebook 01
(sección 6.5). Hace falta una cuenta gratuita de NASA Earthdata guardada en `~/.netrc`
(`machine urs.earthdata.nasa.gov login <usuario> password <clave>`), que `requests` usa sola. Cada archivo pesa 3.5-15 GB
porque trae la franja completa de unos 70 km; lo que se está bajando queda como `.parcial` y se retoma donde quedó.

In [ ]:
nisar = {"083_D_095": ["2025-11-03", "2025-11-27", "2026-01-14", "2026-07-01", "2026-07-13", "2026-07-25", "2026-08-18",
                       "2026-08-30", "2026-10-05", "2026-10-17"],
         "003_A_171": ["2025-10-29", "2026-08-25", "2026-09-06", "2026-09-30", "2026-10-12", "2026-10-24", "2026-11-05"]}
urls = []
for pista, dias in nisar.items():
    for d in dias:
        q = {"platform": "NISAR", "processingLevel": "RSLC", "intersectsWith": "POINT(-76.9435 -12.1804)",
             "start": f"{d}T00:00:00Z", "end": f"{d}T23:59:59Z", "output": "geojson"}
        hallados = requests.get("https://api.daac.asf.alaska.edu/services/search/param", params=q).json()["features"]
        urls += [h["properties"]["url"] for h in hallados if pista in h["properties"]["fileID"]][:1]


def bajar_nisar(url):
    ruta = Path("Data/crudos_nisar/lima") / url.split("/")[-1]
    if ruta.exists():
        return
    ruta.parent.mkdir(parents=True, exist_ok=True)
    parcial = ruta.with_suffix(".h5.parcial")
    hecho = parcial.stat().st_size if parcial.exists() else 0
    with requests.get(url, headers={"Range": f"bytes={hecho}-"}, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(parcial, "ab" if r.status_code == 206 else "wb") as f:
            for bloque in r.iter_content(8 << 20):
                f.write(bloque)
    parcial.rename(ruta)


with ThreadPoolExecutor(3) as ex:
    list(ex.map(bajar_nisar, urls))
print(len(urls), "imágenes NISAR")